In [ ]:
import openai
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import time
import re

# Set API key
openai.api_key = "your-api-key-here"

# Define prompt template
def generate_prompt(comment):
    return f"""
Classify the following company comment about PFAS regulations into one of these categories:

1. **support**: Explicitly endorses the PFAS ban or restriction and provides supporting evidence
2. **against**: Strongly argues against the PFAS ban, citing economic impacts or lack of alternatives
3. **exemption**: Requests specific exemptions or transitional periods
4. **uncertain**: Provides neutral factual information without clear stance

Comment:
\"\"\"{comment}\"\"\"

Respond with only the classification category (support, against, exemption, or uncertain) and a confidence score from 0.0 to 1.0 in this format:
Classification: [category]
Confidence: [score]
"""

# Query GPT-4
def classify_with_gpt(comment, max_retries=3):
    """Classify comment using GPT-4 with retry logic"""
    prompt = generate_prompt(comment)
    
    for attempt in range(max_retries):
        try:
            response = openai.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=0
            )
            
            result_text = response.choices[0].message.content.strip()
            
            # Parse the response
            stance_match = re.search(r'Classification:\s*(\w+)', result_text, re.IGNORECASE)
            confidence_match = re.search(r'Confidence:\s*([\d.]+)', result_text, re.IGNORECASE)
            
            stance = stance_match.group(1).lower() if stance_match else 'uncertain'
            confidence = float(confidence_match.group(1)) if confidence_match else 0.5
            
            # Validate stance
            valid_stances = ['support', 'against', 'exemption', 'uncertain']
            if stance not in valid_stances:
                stance = 'uncertain'
                confidence = 0.3
            
            return {
                'stance': stance,
                'confidence_score': confidence,
                'raw_response': result_text
            }
            
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2)  # Wait before retry
            else:
                return {
                    'stance': 'uncertain',
                    'confidence_score': 0.0,
                    'raw_response': f"Error: {str(e)}"
                }

def apply_gpt_classification_to_dataframe(df, text_column='combined_text', batch_size=5):
    """Apply GPT classification to dataframe with rate limiting"""
    results = []
    
    for i in tqdm(range(0, len(df), batch_size)):
        batch = df.iloc[i:i+batch_size]
        batch_results = []
        
        for text in batch[text_column]:
            result = classify_with_gpt(text)
            batch_results.append(result)
            time.sleep(1)  # Rate limiting - adjust as needed
        
        results.extend(batch_results)
    
    df_classified = df.copy()
    df_classified['stance'] = [r['stance'] for r in results]
    df_classified['confidence_score'] = [r['confidence_score'] for r in results]
    df_classified['gpt_raw_response'] = [r['raw_response'] for r in results]
    
    return df_classified

def perform_gpt_validation_checks(df, text_column='combined_text'):
    """Perform validation checks on GPT classification results"""
    
    # 1. Distribution of stances
    stance_distribution = df['stance'].value_counts()
    print("\nGPT Stance Distribution:")
    print(stance_distribution)

    # 2. Confidence score distribution
    print("\nGPT Confidence Score Statistics:")
    print(df['confidence_score'].describe())

    # 3. Check for potential misclassifications
    low_confidence = df[df['confidence_score'] < 0.4]
    print(f"\nNumber of low confidence GPT classifications: {len(low_confidence)}")

    # 4. Analyze text length vs confidence
    df['text_length'] = df[text_column].str.len()
    correlation = df['text_length'].corr(df['confidence_score'])
    print(f"\nCorrelation between text length and GPT confidence: {correlation:.3f}")

    # 5. Plot confidence distribution by stance
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='stance', y='confidence_score', data=df)
    plt.title('GPT Confidence Score Distribution by Stance')
    plt.show()

# --- Main script starts here ---

# Load your df_translated DataFrame
df_translated = pd.read_csv('df_translated.csv')
print(f"df_translated loaded successfully. Shape: {df_translated.shape}")

# Ensure 'combined_text' column exists
if 'combined_text' not in df_translated.columns:
    raise KeyError("The 'combined_text' column is required but not found in df_translated.")

# Take a sample of 20 for GPT analysis
sample_size = 20
df_sample_gpt = df_translated.sample(n=sample_size, random_state=42).copy()

print(f"\nStarting GPT classification on {sample_size} samples...")

# Apply GPT classification
df_gpt_classified = apply_gpt_classification_to_dataframe(df_sample_gpt)

# Perform validation checks
print("\nPerforming validation checks on GPT results:")
perform_gpt_validation_checks(df_gpt_classified)

# --- Visualizations (same format as zero-shot method) ---

print("\nGenerating GPT classification visualizations...")

# 1. Stance distribution bar chart
plt.figure(figsize=(10, 6))
stance_counts = df_gpt_classified['stance'].value_counts()
sns.barplot(x=stance_counts.index, y=stance_counts.values, palette='viridis')
plt.title('GPT Stance Distribution')
plt.xlabel('Stance')
plt.ylabel('Number of Entries')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 2. Pie chart for stance distribution
plt.figure(figsize=(8, 8))
plt.pie(stance_counts, labels=stance_counts.index, autopct='%1.1f%%', 
        colors=sns.color_palette('viridis', len(stance_counts)))
plt.title('GPT Stance Distribution')
plt.show()

# 3. Confidence score box plot by stance
plt.figure(figsize=(10, 6))
sns.boxplot(x='stance', y='confidence_score', data=df_gpt_classified)
plt.title('GPT Confidence Score Distribution by Stance')
plt.xlabel('Stance')
plt.ylabel('Confidence Score')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 4. Confident classifications bar chart
confident_count = len(df_gpt_classified[df_gpt_classified['stance'] != 'uncertain'])
uncertain_count = len(df_gpt_classified[df_gpt_classified['stance'] == 'uncertain'])

plt.figure(figsize=(8, 6))
categories = ['Confident', 'Uncertain']
counts = [confident_count, uncertain_count]
bars = plt.bar(categories, counts, color=sns.color_palette('viridis', 2), width=0.6)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{int(height)}',
             ha='center', va='bottom', fontsize=12)

plt.title('GPT Confident vs Uncertain Classifications')
plt.ylabel('Number of Classifications')
plt.ylim(0, max(counts) + 2)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 5. Confidence score histogram
plt.figure(figsize=(10, 6))
plt.hist(df_gpt_classified['confidence_score'], bins=10, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('GPT Confidence Score Distribution')
plt.xlabel('Confidence Score')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Display results summary
print("\nGPT Classification Results Summary:")
print(f"Total samples processed: {len(df_gpt_classified)}")
print(f"Confident classifications: {confident_count} ({confident_count/len(df_gpt_classified)*100:.1f}%)")
print(f"Uncertain classifications: {uncertain_count} ({uncertain_count/len(df_gpt_classified)*100:.1f}%)")
print(f"Average confidence score: {df_gpt_classified['confidence_score'].mean():.3f}")

print("\nStance breakdown:")
for stance, count in stance_counts.items():
    percentage = count / len(df_gpt_classified) * 100
    avg_confidence = df_gpt_classified[df_gpt_classified['stance'] == stance]['confidence_score'].mean()
    print(f"  {stance}: {count} ({percentage:.1f}%) - Avg confidence: {avg_confidence:.3f}")

# Save results
output_filename = 'df_translated_sample_gpt_sentiment.csv'
df_gpt_classified.to_csv(output_filename, index=False, encoding='utf-8')
print(f"\nGPT classification results saved to '{output_filename}'")

# Display sample results
print("\nSample GPT Classification Results:")
display_cols = ['combined_text', 'stance', 'confidence_score']
print(df_gpt_classified[display_cols].head().to_string())